# Phân tích dữ liệu DUC_SUM

Notebook thống kê số file không có nội dung và số lượng thẻ mở `<s>` trong từng file.

## Tóm tắt

Chạy toàn bộ notebook để cập nhật kết quả theo dữ liệu hiện có trong `data/DUC_SUM`.

## Bối cảnh và phương pháp

### Giả định chính

- Một file được xem là **không có nội dung** nếu sau khi đọc và loại bỏ khoảng trắng, nội dung còn lại là chuỗi rỗng.
- Số tag `<s>` được tính bằng số thẻ mở có dạng `<s ...>` (không phân biệt chữ hoa/thường).
- Trung bình, lớn nhất và nhỏ nhất được tính trên **tất cả file**, bao gồm cả file rỗng. Notebook cũng hiển thị trung bình riêng trên các file có nội dung để tiện đối chiếu.

In [ ]:
from pathlib import Path
import re
import statistics

def find_project_root(start_path):
    current_path = start_path.resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if (candidate_path / 'data' / 'DUC_SUM').is_dir():
            return candidate_path
    raise FileNotFoundError('Không tìm thấy thư mục data/DUC_SUM.')

project_root = find_project_root(Path.cwd())
duc_sum_dir = project_root / 'data' / 'DUC_SUM'
print(f'Thư mục dữ liệu: {duc_sum_dir}')

## Dữ liệu

Đọc toàn bộ file trực tiếp trong `data/DUC_SUM`, sắp xếp theo tên để kết quả luôn ổn định.

In [ ]:
duc_sum_files = sorted(path for path in duc_sum_dir.iterdir() if path.is_file())

file_statistics = []
for file_path in duc_sum_files:
    file_content = file_path.read_text(encoding='utf-8', errors='replace')
    has_content = bool(file_content.strip())
    sentence_tag_count = len(re.findall(r'<s\b', file_content, flags=re.IGNORECASE))

    file_statistics.append({
        'file_name': file_path.name,
        'has_content': has_content,
        'sentence_tag_count': sentence_tag_count,
    })

print(f'Đã đọc {len(file_statistics)} file.')

## Kết quả

In [ ]:
if not file_statistics:
    raise ValueError(f'Không có file nào trong {duc_sum_dir}.')

empty_files = [row for row in file_statistics if not row['has_content']]
all_tag_counts = [row['sentence_tag_count'] for row in file_statistics]
nonempty_tag_counts = [
    row['sentence_tag_count'] for row in file_statistics if row['has_content']
]
files_with_sentence_tags = [
    row for row in file_statistics if row['sentence_tag_count'] > 0
]

total_tag_count = sum(all_tag_counts)
average_tag_count = total_tag_count / len(all_tag_counts)
median_tag_count = statistics.median(all_tag_counts)
quartiles = statistics.quantiles(all_tag_counts, n=4, method='inclusive')
first_quartile_tag_count = quartiles[0]
third_quartile_tag_count = quartiles[2]
tag_count_standard_deviation = statistics.pstdev(all_tag_counts)
maximum_tag_count = max(all_tag_counts)
minimum_tag_count = min(all_tag_counts)
average_nonempty_tag_count = (
    sum(nonempty_tag_counts) / len(nonempty_tag_counts)
    if nonempty_tag_counts else 0
)
minimum_nonempty_tag_count = min(nonempty_tag_counts) if nonempty_tag_counts else 0
maximum_nonempty_tag_count = max(nonempty_tag_counts) if nonempty_tag_counts else 0

maximum_files = [
    row['file_name'] for row in file_statistics
    if row['sentence_tag_count'] == maximum_tag_count
]
minimum_files = [
    row['file_name'] for row in file_statistics
    if row['sentence_tag_count'] == minimum_tag_count
]

print(f'Tổng số file: {len(file_statistics)}')
print(f'Số file không có nội dung: {len(empty_files)}')
print(f'Số file có nội dung: {len(file_statistics) - len(empty_files)}')
print(f'Số file có ít nhất một tag <s>: {len(files_with_sentence_tags)}')
print(f'Tổng số tag <s>: {total_tag_count}')
print(f'Số tag <s> trung bình (tất cả file): {average_tag_count:.2f}')
print(f'Số tag <s> trung bình (file có nội dung): {average_nonempty_tag_count:.2f}')
print(f'Trung vị số tag <s>: {median_tag_count:.2f}')
print(f'Tứ phân vị Q1: {first_quartile_tag_count:.2f}')
print(f'Tứ phân vị Q3: {third_quartile_tag_count:.2f}')
print(f'Độ lệch chuẩn số tag <s>: {tag_count_standard_deviation:.2f}')
maximum_file_names = ', '.join(maximum_files)
minimum_file_names = ', '.join(minimum_files)
print(f'Số tag <s> nhiều nhất: {maximum_tag_count} - file: {maximum_file_names}')
print(f'Số tag <s> thấp nhất: {minimum_tag_count} - file: {minimum_file_names}')
print(f'Số tag <s> thấp nhất trong file có nội dung: {minimum_nonempty_tag_count}')
print(f'Số tag <s> nhiều nhất trong file có nội dung: {maximum_nonempty_tag_count}')

### Danh sách file không có nội dung

In [ ]:
if empty_files:
    for row in empty_files:
        print(f"- {row['file_name']}")
else:
    print('Không có file rỗng.')

### Chi tiết số tag `<s>` theo file

In [ ]:
print(f"{'Tên file':<12} {'Có nội dung':<14} {'Số tag <s>':>10}")
print('-' * 38)
for row in file_statistics:
    content_status = 'Có' if row['has_content'] else 'Không'
    print(f"{row['file_name']:<12} {content_status:<14} {row['sentence_tag_count']:>10}")

## Kết luận

Kết quả phía trên được tính trực tiếp từ dữ liệu mỗi lần chạy notebook. Giá trị nhỏ nhất trên tất cả file có thể bằng `0` vì các file không có nội dung vẫn thuộc tập dữ liệu phân tích.